In [ ]:
# for linux
# !apt-get install poppler-utils tesseract-ocr libmagic-dev

# for mac
# !brew install poppler tesseract libmagic

In [1]:
%pip install -Uq "unstructured[all-docs]" 
%pip install -Uq langchain_chroma 
%pip install pymupdf
%pip install -Uq langchain langchain-community langchain-openai 
%pip install -Uq python_dotenv
%pip install langchain-groq


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note

In [2]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

/Users/yifeishang/Desktop/desk/RAG/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
# import fitz  # pip install pymupdf
# import base64
# file_path = "./docs/2_Sikafloor-263 SL.pdf"


# def has_text_layer(pdf_path: str, sample_pages: int = 5) -> bool:
#     """检查 PDF 是否有可提取的文字层（对比扫描版）"""
#     doc = fitz.open(pdf_path)
#     total = min(sample_pages, len(doc))
#     char_count = sum(len(doc[i].get_text()) for i in range(total))
#     doc.close()
#     return char_count > 100 * total  # 平均每页超过100字符认为有文字层


# def extract_titles_by_fontsize(pdf_path: str, min_font_size: float = 13.0) -> dict:
#     """方案 B：通过字体大小提取每页标题，返回 {page_num: title}"""
#     doc = fitz.open(pdf_path)
#     page_titles = {}
    
#     for page_num in range(len(doc)):
#         page = doc[page_num]
#         blocks = page.get_text('dict')['blocks']
#         candidates = []
        
#         for block in blocks:
#             for line in block.get('lines', []):
#                 for span in line.get('spans', []):
#                     text = span['text'].strip()
#                     size = span['size']
#                     if size >= min_font_size and text and len(text) > 2:
#                         candidates.append((size, text))
        
#         if candidates:
#             # 取字体最大的那个作为标题
#             candidates.sort(key=lambda x: -x[0])
#             page_titles[page_num + 1] = candidates[0][1]
    
#     doc.close()
#     return page_titles


# def extract_title_by_vision(pdf_path: str, page_num: int) -> str:
#     """方案 A：用 GPT-4o 视觉识别单页的产品名称"""
#     doc = fitz.open(pdf_path)
#     page = doc[page_num - 1]
#     pix = page.get_pixmap(dpi=150)
#     img_b64 = base64.b64encode(pix.tobytes('jpeg')).decode()
#     doc.close()
    
#     llm = ChatOpenAI(model='gpt-4o', temperature=0)
#     msg = HumanMessage(content=[
#         {"type": "text", "text": "This is a building materials product catalogue page. What is the main product name or heading on this page? Reply with just the product name, nothing else. If there is no clear product name, reply 'UNKNOWN'."},
#         {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
#     ])
#     return llm.invoke([msg]).content.strip()


# def get_page_titles(pdf_path: str) -> dict:
#     """自动判断：有文字层用方案B（字体大小），没有用方案A（GPT-4o视觉）"""
#     print('🔍 Checking PDF text layer...')
    
#     if has_text_layer(pdf_path):
#         print('✅ Text layer found → using font size extraction (fast)')
#         titles = extract_titles_by_fontsize(pdf_path)
#     else:
#         print('⚠️  No text layer → using GPT-4o vision (slower)')
#         doc = fitz.open(pdf_path)
#         titles = {}
#         for page_num in range(1, len(doc) + 1):
#             print(f'   Analyzing page {page_num}/{len(doc)}...')
#             titles[page_num] = extract_title_by_vision(pdf_path, page_num)
#         doc.close()
    
#     print(f'✅ Extracted titles for {len(titles)} pages')
#     return titles


# # 运行：获取每页的标题
# page_titles = get_page_titles(file_path)

# # 查看结果（只打印有内容的页）
# for page, title in sorted(page_titles.items()):
#     print(f'  Page {page:3d}: {title}')

In [4]:
# def partition_document(file_path: str):
#     """Extract elements from PDF using unstructured"""
#     print(f"📄 Partitioning document: {file_path}")
    
#     elements = partition_pdf(
#         filename=file_path,  # Path to your PDF file
#         strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
#         infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
#         extract_image_block_types=[], # Grab images found in the PDF
#         extract_image_block_to_payload=False # Store images as base64 data you can actually use
#     )
    
#     print(f"✅ Extracted {len(elements)} elements")
#     return elements

# # Test with your PDF file
#   # Change this to your PDF path
# elements = partition_document(file_path)

In [5]:
# All types of different atomic elements we see from unstructured
# set([str(type(el)) for el in elements])

In [6]:
# def create_chunks_by_title(elements, page_titles: dict = None):
#     """Create intelligent chunks using title-based strategy"""
#     print("🔨 Creating smart chunks...")
    
#     chunks = chunk_by_title(
#         elements,
#         max_characters=3000,
#         new_after_n_chars=2400,
#         combine_text_under_n_chars=500
#     )
    
#     print(f"✅ Created {len(chunks)} chunks")
    
#     # 提取构件名和对应表格
#     components = []
#     for i, chunk in enumerate(chunks):
#         component_name = None
#         chunk_pages = set()
#         tables = []
#         if hasattr(chunk.metadata, 'orig_elements'):
#             for el in chunk.metadata.orig_elements:
#                 el_type = type(el).__name__
#                 if el_type == 'Title':
#                     component_name = el.text.strip()  # 每次遇到 Title 都覆盖，最终保留最后一个
#                 elif el_type == 'Table':
#                     tables.append(getattr(el.metadata, 'text_as_html', el.text))
#                 # 收集这个 chunk 涉及的页码
#                 page = getattr(el.metadata, 'page_number', None)
#                 if page:
#                     chunk_pages.add(page)
#         # 如果没有 Title，用 page_titles 补全（取 chunk 第一页的标题）
#         if not component_name and page_titles and chunk_pages:
#             first_page = min(chunk_pages)
#             component_name = page_titles.get(first_page)
#         if component_name:
#             components.append({'chunk_id': i, 'component_name': component_name, 'tables': tables})
    
#     print(f"✅ Found {len(components)} components")
#     for c in components:
#         print(f"  [{c['chunk_id']}] {c['component_name']} — {len(c['tables'])} 张表")
    
#     return chunks, components

# # Create chunks，传入 page_titles 补全被 OCR 漏掉的标题
# chunks, components = create_chunks_by_title(elements, page_titles=page_titles)

In [ ]:
import fitz
import base64
import json
import os
file_path = "./docs/expanded-metal-catalogue.pdf"

def page_to_base64(pdf_path: str, page_num: int, dpi: int = 150) -> str:
    """Convert a PDF page to base64 JPEG image"""
    doc = fitz.open(pdf_path)
    page = doc[page_num - 1]
    pix = page.get_pixmap(dpi=dpi)
    img_b64 = base64.b64encode(pix.tobytes('jpeg')).decode()
    doc.close()
    return img_b64


def extract_page_catalogue(pdf_path, page_num, llm, context_products=None) -> dict:
    """Ask GPT-4o to extract product catalogue data from a single page"""
    img_b64 = page_to_base64(pdf_path, page_num)
    context_hint = ""
    context_hint = ""
    if context_products:
        products_str = ', '.join(f'"{p}"' for p in context_products)
        context_hint = f"""

    Note: The previous page introduced product(s): {products_str}.
    If any heading on THIS page looks like a SECTION HEADER rather than a product name
    (e.g. "PRODUCT INFORMATION", "TECHNICAL INFORMATION", "APPLICATION INFORMATION",
    "SYSTEM INFORMATION", "BASIS OF PRODUCT DATA", "MAINTENANCE", "ECOLOGY", etc.),
    do NOT use it as product_name — set product_name to null instead.
    The calling code will automatically assign items to the correct product.
    """
    prompt = """This is a page from a building materials product catalogue.

Extract ALL product data from this page and return a JSON ARRAY. Each element represents one product section:
[
  {
    \"product_name\": \"ARCH LINTELS\",
    \"description\": \"brief product description\",
    \"items\": [
      {\"item_number\": \"SMAB_F_00054403\", \"attribute1\": \"value1\", \"attribute2\": \"value2\"},
      ...
    ]
  },
  {
    \"product_name\": \"LINTELS BRACKETS\",
    \"description\": \"...\",
    \"items\": [...]
  }
]

Rules:
- If the page is a cover/title page with a prominent product or brand name but no table, return: [{"product_name": "THE TITLE", "description": "...", "items": []}]
- Only return [] if the page has absolutely no useful information (blank, pure legal text, etc.)
- Use the exact column names from each table as attribute keys
- product_name MUST be a large heading/title visibly printed on THIS page. Do NOT guess or infer from content.
- If there is no clear large title, set product_name to null
- A product entry with only a title and no table items is valid: {\"product_name\": \"TITLE\", \"description\": \"...\", \"items\": []}
- If there is table data but no clear title, extract items with product_name set to null
- Only return [] if the page truly has NO table data at all (cover page, index page, etc.)
- Return ONLY a valid JSON array, no extra text""" + context_hint

    msg = HumanMessage(content=[
        {"type": "text", "text": prompt},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
    ])
    
    response = llm.invoke([msg]).content.strip()
    if response.startswith('```'):
        response = response.split('```')[1]
        if response.startswith('json'):
            response = response[4:]
    try:
        parsed = json.loads(response)
        return parsed if isinstance(parsed, list) else [parsed]
    except json.JSONDecodeError:
        return []

from langchain_groq import ChatGroq

def build_catalogue(pdf_path: str, start_page: int = 1, end_page: int = None) -> dict:
    """逐页用 GPT-4o 提取产品目录。当某页没有标题时，沿用上一页的类目"""
    # llm = ChatOpenAI(model='gpt-4o', temperature=0)
    llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    doc.close()

    end_page = end_page or total_pages
    catalogue = {}
    last_product_name = None   # 最近一次成功归类的类目
    prev_product_names = []    # 前一页 GPT 返回的所有标题（未经回退）
    print(f'Processing pages {start_page} to {end_page} of {total_pages}...')

    for page_num in range(start_page, end_page + 1):
        print(f'  Page {page_num}/{total_pages}...', end=' ')
        results = extract_page_catalogue(pdf_path, page_num, llm, context_products=prev_product_names if prev_product_names else None)
                                

        page_had_data = False
        cur_product_names = []  # 本页 GPT 返回的所有标题

        for entry in results:
            product_name = entry.get('product_name')
            items = entry.get('items', [])

            if product_name:
                cur_product_names.append(product_name)

            # 本条目没有标题但有数据 → 找回退类目
            if not product_name and items:
                if prev_product_names:
                    product_name = prev_product_names[-1]
                    print(f'\u21a9\ufe0f  用前一页标题: {product_name}', end=' ')
                elif last_product_name:
                    product_name = last_product_name
                    print(f'\u21a9\ufe0f  沿用上一类目: {product_name}', end=' ')

            if product_name and items:
                if product_name not in catalogue:
                    catalogue[product_name] = {'description': entry.get('description', ''), 'items': []}
                existing_ids = {item.get('item_number') for item in catalogue[product_name]['items']}
                new_items = [item for item in items if item.get('item_number') not in existing_ids]
                catalogue[product_name]['items'].extend(new_items)
                last_product_name = product_name
                print(f'\u2705 {product_name} ({len(new_items)} items)', end=' ')
                page_had_data = True
            elif product_name and not items:
                if product_name not in catalogue:
                    catalogue[product_name] = {'description': entry.get('description', ''), 'items': []}
                # title-only 页：记录标题，等下一页数据来
                print(f'\U0001f4cc {product_name} (title only)', end=' ')
                page_had_data = True

        if not page_had_data:
            print('skipped', end='')
        print()

        prev_product_names = cur_product_names  # 更新前一页标题列表

        # 每页完成后写入 JSON，前端刷新即可看到最新数据
        with open('gpt_catalogue_progress.json', 'w', encoding='utf-8') as f:
            json.dump(catalogue, f, ensure_ascii=False)

    print(f'\n\u2705 Done! Found {len(catalogue)} products')
    return catalogue

gpt_catalogue = build_catalogue(file_path, start_page=1, end_page=10)

for product_name, data in gpt_catalogue.items():
    print(f'\n=== {product_name} ({len(data["items"])} items) ===')
    print(f'  Description: {data["description"]}')
    for item in data['items']:
        print(f'  {item}')

Processing pages 1 to 10 of 39...
  Page 1/39... 

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.2-90b-vision-instruct` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [ ]:
import json, os, threading, webbrowser
from http.server import HTTPServer, SimpleHTTPRequestHandler

# 1. Export catalogue to JSON (re-run anytime to refresh data)
with open('catalogue_data.json', 'w', encoding='utf-8') as f:
    json.dump(gpt_catalogue, f, indent=2, ensure_ascii=False)
print('✅ Exported catalogue_data.json')

# 2. Start local HTTP server (needed so fetch() can load the JSON)
PORT = 8765

class SilentHandler(SimpleHTTPRequestHandler):
    def log_message(self, format, *args): pass  # suppress server logs

def start_server():
    os.chdir(os.path.dirname(os.path.abspath('frontend.html')))
    server = HTTPServer(('localhost', PORT), SilentHandler)
    server.serve_forever()

# Only start if not already running
try:
    import socket
    s = socket.socket()
    s.connect(('localhost', PORT))
    s.close()
    print(f'✅ Server already running on port {PORT}')
except ConnectionRefusedError:
    t = threading.Thread(target=start_server, daemon=True)
    t.start()
    print(f'✅ Server started on port {PORT}')

# 3. Open browser
url = f'http://localhost:{PORT}/frontend.html'
webbrowser.open(url)
print(f'🌐 Opening {url}')
print('   (Re-run this cell after updating catalogue_data.json to reload data)')

✅ Exported catalogue_data.json
✅ Server already running on port 8765
🌐 Opening http://localhost:8765/frontend.html
   (Re-run this cell after updating catalogue_data.json to reload data)
